# Creator Earnings — OLS Diagnostics

Reproducible diagnostics for the AdSense-equivalent earnings estimator
(`apps/ml/earnings.py`). The target is `log(monthly_views x niche_CPM / 1000 x 0.55)`
plus income noise standing in for unobserved brand-deal / merch / membership
revenue. Trained on a **simulated 1000-creator cohort**; genuine extraction over
the 52 live channels is the inference path. Headline metric: **5-fold CV R^2 ~ 0.67**
(held-out 20% split is noisier). See `docs/decisions.md` ADR-0014.

Run from `analysis/notebooks/` with the venv active and `.env` sourced
(needs `DATABASE_URL`).

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../.."))
import matplotlib.pyplot as plt
import statsmodels.api as sm

from apps.ml import earnings as E

eng = E.get_engine()
real = E.load_real(eng)
sim = E.simulate_cohort(real)
niches = sorted(real["niche"].dropna().unique().tolist())
print(f"live channels={len(real)}  simulated cohort={len(sim)}  niches={len(niches)}")

## OLS fit + coefficient table (with 95% CIs)

In [ ]:
x, names = E.build_design(sim, niches)
y = sim["log_earnings"].to_numpy()
ntr = int(len(sim) * 0.8)
res = sm.OLS(y[:ntr], x[:ntr]).fit()
res.summary(xname=names)

**Reading the coefficients:** `log_monthly_views` should sit near 1.0 (the
target is ~linear in log views); the `niche_*` dummies recover each niche's CPM
offset relative to the dropped baseline niche. `log_subscribers`, engagement and
cadence carry little signal once views are in — expected, since the CPM target
does not depend on them.

## Held-out vs 5-fold CV R^2

In [ ]:
pred = res.predict(x[ntr:]); yt = y[ntr:]
r2_holdout = 1 - ((yt - pred)**2).sum() / ((yt - yt.mean())**2).sum()
print(f"held-out R^2 = {r2_holdout:.3f}   (single 20% split — noisy)")
print(f"5-fold CV R^2 = {E.cv_r2(x, y):.3f}   (headline)")
print(f"train R^2 = {res.rsquared:.3f}")

## Residual diagnostics

In [ ]:
fitted = res.predict(x[:ntr]); resid = y[:ntr] - fitted
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(fitted, resid, s=12, alpha=0.6); ax.axhline(0, color="red", lw=1)
ax.set_xlabel("Fitted"); ax.set_ylabel("Residuals")
ax.set_title("Residuals vs Fitted (simulated cohort)")
plt.show()

In [ ]:
sm.qqplot(resid, line="45", fit=True)
plt.title("Normal Q-Q (simulated cohort)")
plt.show()

**Limitations.** Earnings are AdSense-equivalent only (no brand deals, merch,
memberships, Super Chats). Labels are CPM-derived, not real income; the noise term
is a stand-in, so R^2 reflects that injected difficulty, not validated income
prediction. Numbers are defensible only as *validated against a simulated cohort*.